In [ ]:
pip install llama-index llama-index-llms-openai python-dotenv tiktoken colorama nltk

Install Langsmith SDK

In [ ]:
pip install -U openai langsmith

## Setup

**Loading Environment Variables**

In [ ]:
from getpass import getpass
from dotenv import load_dotenv

import os

load_dotenv(override=True)  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
    

**OpenAI Secret Key**

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

embed_model = OpenAIEmbedding(
    api_key=os.getenv("OPENAI_API_KEY")
)
# -----------------------------
# LLM Setup
# -----------------------------
llm = OpenAI(model="gpt-4o-mini", temperature=0)

**Tracing System**

In [ ]:

# -----------------------------
# TRACING SYSTEM (LangSmith-style)
# -----------------------------
import uuid

class Tracer:
    def __init__(self):
        self.trace_id = str(uuid.uuid4())

    def log(self, step, data):
        print(f"\n🔍 TRACE [{self.trace_id}] - {step}")
        print(data)

tracer = Tracer()

In [ ]:
# -----------------------------
# DATA INGESTION
# ----------------------------

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/refs/heads/main/docs/examples/data/paul_graham/paul_graham_essay_ambiguated.txt' -O 'data/paul_graham/paul_graham_essay-ambiguated.txt'

In [ ]:
import os, json, math, time
from datetime import datetime, timedelta
from typing import Literal, Optional
from enum import Enum
from dataclasses import dataclass, field
from pydantic import BaseModel, Field
from collections import defaultdict

from llama_index.core import (
    Settings, VectorStoreIndex, SimpleDirectoryReader,
    StorageContext, load_index_from_storage,
)
from llama_index.core.schema import NodeWithScore 
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from colorama import Fore, Style
from langsmith import traceable


Settings.llm       = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
llm = Settings.llm

# =====================================================
# BUILD OR LOAD INDEX
# =====================================================

STORAGE_DIR = "./storage"
DATA_DIR    = "./data/paul_graham"       # folder with your .txt / .pdf files

if not os.path.exists(STORAGE_DIR):
    print(f"{Fore.GREEN}Building index for the first time...{Style.RESET_ALL}")

    # Load documents
    documents = SimpleDirectoryReader(DATA_DIR).load_data()
    print(f"Loaded {len(documents)} document(s)")

    # Chunk semantically
    splitter = SemanticSplitterNodeParser(embed_model=Settings.embed_model)
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"Created {len(nodes)} nodes")

    # Build + persist
    index = VectorStoreIndex(nodes)
    index.storage_context.persist(STORAGE_DIR)
    print(f"Index saved to {STORAGE_DIR}/")

else:
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
    index = load_index_from_storage(storage_context)
    print("Index loaded.")

# Retrieve all nodes for BM25
all_nodes = list(index.docstore.docs.values())
print(f"Total nodes available for BM25: {len(all_nodes)}")

In [ ]:
# -----------------------------
# QUERY TRANSFORMATION : Query Rewriting / Decomposition / Classification / Routing 
# -----------------------------

REWRITE_SYSTEM = """You are a query optimization expert. Rewrite the user's query to be more
specific, clear, and optimized for semantic search retrieval.
Rules: expand abbreviations, add context terms, make implicit concepts explicit.
Return ONLY the rewritten query."""

HYDE_SYSTEM = """You are an expert writer.

Given a user question, generate a detailed, high-quality answer 
as if you already knew the answer.

This answer will be used for semantic search retrieval.

Be informative, specific, and include relevant concepts.
Do NOT say "I don't know".

Return ONLY the generated answer."""

DECOMPOSE_SYSTEM = """Break the query into 2-4 simpler sub-questions.
Return a JSON array of strings only."""

STEPBACK_SYSTEM = """Generate a more general step-back question for the specific query.
Return ONLY the step-back question."""

ROUTER_SYSTEM = """Classify the query:
- factual    Simple fact lookup
- analytical Complex multi-fact reasoning
- conceptual Abstract / how-why
- keyword    Exact terms / proper nouns
Return JSON: {"query_type": "...", "reasoning": "..."}"""

def rewrite_query(q):
    return llm.complete(f"{REWRITE_SYSTEM}\n\nOriginal: {q}").text.strip()

def hyde_generate(query: str) -> str:
    response = llm.complete(f"{HYDE_SYSTEM}\n\nQuestion: {query}")
    return response.text.strip()

def decompose_query(q):
    raw = llm.complete(f"{DECOMPOSE_SYSTEM}\n\nQuery: {q}").text.strip()
    return json.loads(raw.removeprefix("```json").removesuffix("```").strip())

def stepback_query(q):
    return llm.complete(f"{STEPBACK_SYSTEM}\n\nQuery: {q}").text.strip()

class QueryClassification(BaseModel):
    query_type: Literal["factual","analytical","conceptual","keyword"]
    reasoning: str

def classify_query(q):
    raw  = llm.complete(f"{ROUTER_SYSTEM}\n\nQuery: {q}").text.strip()
    data = json.loads(raw.removeprefix("```json").removesuffix("```").strip())
    return QueryClassification(**data)

In [ ]:
def classify_and_transform(query, verbose=False):
    """Classifies the query and applies appropriate transformations based on the classification."""
    classification = classify_query(query)
    if verbose:
        print(f"[Router]     {classification.query_type} — {classification.reasoning}")
        
        # Step 2 — optionally rewrite / decompose / step-back *(replace with HyDE for factual)
        if classification.query_type == "factual":
            retrieval_query = rewrite_query(query)
            # hyde_doc = hyde_generate(retrieval_query)
            if verbose: 
                print(f"[Rewrite]   {retrieval_query}")
                # print(f"[HyDE] Generated doc:\n{hyde_doc[:200]}...\n")
        
            

        elif classification.query_type in ("analytical", "conceptual"):
            sub_qs = decompose_query(query)
            retrieval_query = " ".join(sub_qs)   # join sub-questions for broader retrieval
            if verbose: print(f"[Decompose]  {sub_qs}")

        elif classification.query_type == "keyword":
            retrieval_query = query              # keep exact terms for BM25
            if verbose: print(f"[Keyword]    keeping original")

        else:
            retrieval_query = query
        return retrieval_query, classification 

## Hybrid Search (Vector + Keyword)

In [ ]:
# =====================================================
# ADAPTIVE RETRIEVAL (BM25 + Vector Search + Reranking (RRF) + Fusion)
# =====================================================

# --- Load existing index ---
from llama_index.core import StorageContext, load_index_from_storage
storage_context = StorageContext.from_defaults(persist_dir="./storage")
index = load_index_from_storage(storage_context)

# Retrieve all nodes for BM25
all_nodes = list(index.docstore.docs.values())

# Retriever weights by query type
RETRIEVER_WEIGHTS = {
    "factual":    [0.8, 0.2],  # vector-heavy
    "keyword":    [0.3, 0.7],  # BM25-heavy
    "analytical": [0.5, 0.5],  # balanced
    "conceptual": [0.5, 0.5],  # balanced
}

@traceable()
def get_adaptive_retriever(query_type: str, top_k: int = 5):
    """Returns a fused retriever weighted for the given query type."""
    weights = RETRIEVER_WEIGHTS.get(query_type, [0.6, 0.4])

    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    bm25_retriever   = BM25Retriever.from_defaults(nodes=all_nodes, similarity_top_k=top_k)

    return QueryFusionRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        retriever_weights=weights,
        num_queries=1,       # no extra LLM query generation
        use_async=False,
    )


In [ ]:

# ─────────────────────────────────────────────────────────
#  RECIPROCAL RANK FUSION (RRF) + LEARNED FUSION
# ─────────────────────────────────────────────────────────

def reciprocal_rank_fusion(
    result_lists: list[list[NodeWithScore]],
    k: int = 60,
) -> list[NodeWithScore]:
    """
    Classic RRF: score(d) = Σ 1/(k + rank_i(d))
    k=60 is the standard constant (Cormack et al. 2009).
    Works for any number of retriever result lists.
    """
    scores: dict[str, float]         = defaultdict(float)
    node_map: dict[str, NodeWithScore] = {}

    for results in result_lists:
        for rank, node_with_score in enumerate(results, start=1):
            nid = node_with_score.node.node_id
            scores[nid]   += 1.0 / (k + rank)
            node_map[nid]  = node_with_score

    fused = []
    for nid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        n        = node_map[nid]
        n.score  = score
        fused.append(n)
    return fused


class LearnedFusionWeights(BaseModel):
    """
    Simulate learned fusion: weights are trained offline and
    loaded here. In production, replace with a small regression
    model or a lookup table trained on click/relevance data.
    """
    vector_weight: float = 0.65
    bm25_weight:   float = 0.35

    def blend(
        self,
        vector_results: list[NodeWithScore],
        bm25_results:   list[NodeWithScore],
    ) -> list[NodeWithScore]:
        scores: dict[str, float]           = defaultdict(float)
        node_map: dict[str, NodeWithScore] = {}

        def _add(results, weight):
            max_s = max((n.score or 0) for n in results) if results else 1.0
            for n in results:
                nid           = n.node.node_id
                norm_score    = (n.score or 0) / (max_s or 1)
                scores[nid]  += weight * norm_score
                node_map[nid] = n

        _add(vector_results, self.vector_weight)
        _add(bm25_results,   self.bm25_weight)

        fused = []
        for nid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
            n       = node_map[nid]
            n.score = score
            fused.append(n)
        return fused


def retrieve_with_rrf(query: str, top_k: int = 5) -> list[NodeWithScore]:
    """Run both retrievers independently then fuse with RRF."""
    vec   = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25  = BM25Retriever.from_defaults(
                nodes=all_nodes, similarity_top_k=top_k
            ).retrieve(query)
    return reciprocal_rank_fusion([vec, bm25])[:top_k]


def retrieve_with_learned_fusion(query: str, top_k: int = 5) -> list[NodeWithScore]:
    """Run both retrievers independently then blend with learned weights."""
    vec   = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25  = BM25Retriever.from_defaults(
                nodes=all_nodes, similarity_top_k=top_k
            ).retrieve(query)
    weights = LearnedFusionWeights()          # load from file/DB in production
    return weights.blend(vec, bm25)[:top_k]

In [ ]:

# ─────────────────────────────────────────────────────────
# HYBRID SEARCH PARAMETER TUNING
#   - Static weights  → fixed per query-type (as before)
#   -Dynamic weights → adjusted at query time using query
#                       features (length, has_numbers, etc.)
# ─────────────────────────────────────────────────────────

STATIC_WEIGHTS = {
    "factual":    (0.8, 0.2),
    "analytical": (0.5, 0.5),
    "conceptual": (0.5, 0.5),
    "keyword":    (0.3, 0.7),
}

def _query_features(query: str) -> dict:
    tokens = query.split()
    return {
        "length":      len(tokens),
        "has_numbers": any(t.isdigit() for t in tokens),
        "has_quotes":  '"' in query or "'" in query,
        "is_question": query.strip().endswith("?"),
    }

def dynamic_weights(query: str, query_type: str) -> tuple[float, float]:
    """
    Start from static weights, then nudge based on query features:
      • Short queries (≤3 words)  → boost BM25 (exact match likely)
      • Quoted phrases            → boost BM25 heavily
      • Numbers present           → boost vector (semantic context)
    """
    vec_w, bm25_w = STATIC_WEIGHTS.get(query_type, (0.6, 0.4))
    feats = _query_features(query)

    if feats["has_quotes"]:
        bm25_w = min(bm25_w + 0.20, 0.95)
    if feats["length"] <= 3:
        bm25_w = min(bm25_w + 0.10, 0.95)
    if feats["has_numbers"]:
        vec_w  = min(vec_w  + 0.10, 0.95)

    # re-normalise
    total  = vec_w + bm25_w
    return round(vec_w / total, 3), round(bm25_w / total, 3)


def tuned_hybrid_retrieve(
    query: str,
    query_type: str,
    top_k: int = 5,
    use_dynamic: bool = True,
    verbose: bool = True,
) -> list[NodeWithScore]:
    vec_w, bm25_w = (
        dynamic_weights(query, query_type)
        if use_dynamic else
        STATIC_WEIGHTS.get(query_type, (0.6, 0.4))
    )
    if verbose:
        print(f"[Weights]  vector={vec_w}  bm25={bm25_w}  "
              f"({'dynamic' if use_dynamic else 'static'})")

    vec_results  = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    bm25_results = BM25Retriever.from_defaults(
                       nodes=all_nodes, similarity_top_k=top_k
                   ).retrieve(query)

    # Weighted score blend
    scores: dict[str, float]           = defaultdict(float)
    node_map: dict[str, NodeWithScore] = {}

    def _add(results, weight):
        max_s = max((n.score or 1e-9) for n in results) if results else 1.0
        for n in results:
            nid          = n.node.node_id
            scores[nid] += weight * (n.score or 0) / max_s
            node_map[nid] = n

    _add(vec_results,  vec_w)
    _add(bm25_results, bm25_w)

    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [node_map[nid] for nid, _ in fused[:top_k]]

In [ ]:

# ─────────────────────────────────────────────────────────
# Recency, popularity, user preferences, metadata filters
# ─────────────────────────────────────────────────────────

def _recency_boost(node: NodeWithScore, half_life_days: float = 30.0) -> float:
    """Exponential decay: score × e^(-λ·age).  No metadata → no boost."""
    created = node.node.metadata.get("created_at")
    if not created:
        return 0.0
    try:
        age_days = (datetime.utcnow() - datetime.fromisoformat(created)).days
        lam      = math.log(2) / half_life_days
        return math.exp(-lam * age_days) * 0.15   # max +15% boost
    except Exception:
        return 0.0


def _popularity_boost(node: NodeWithScore) -> float:
    """Linear boost from a 'views' metadata field (0–1 normalised)."""
    views     = float(node.node.metadata.get("views", 0))
    max_views = 10_000.0
    return min(views / max_views, 1.0) * 0.10    # max +10% boost



def _metadata_filter(
    nodes: list[NodeWithScore],
    filters: Optional[dict] = None,
) -> list[NodeWithScore]:
    """
    Hard filter on metadata fields before scoring.
    e.g. filters={"source": "paulgraham.com", "year": "2005"}
    """
    if not filters:
        return nodes
    out = []
    for n in nodes:
        if all(str(n.node.metadata.get(k)) == str(v) for k, v in filters.items()):
            out.append(n)
    return out


def extended_signal_retrieve(
    query: str,
    metadata_filters: Optional[dict] = None,
    top_k: int = 5,
    verbose: bool = True,
) -> list[NodeWithScore]:
    """Retrieve then re-rank with recency + popularity + preference boosts."""
    raw   = index.as_retriever(similarity_top_k=top_k * 3).retrieve(query)
    nodes = _metadata_filter(raw, metadata_filters)

    for n in nodes:
        base   = n.score or 0.0
        boost  = 0.0
        boost += _recency_boost(n)
        boost += _popularity_boost(n)
        n.score = base + boost
        if verbose:
            print(f"  node={n.node.node_id[:8]}  base={base:.3f}  "
                  f"+boost={boost:.3f}  final={n.score:.3f}")

    nodes.sort(key=lambda n: n.score or 0, reverse=True)
    return nodes[:top_k]


In [ ]:
# -----------------------------
# SUMMARIZATION + TRUNCATION
# -----------------------------

import tiktoken

def count_tokens(text):
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    return len(enc.encode(text))

@traceable()
def truncate_by_tokens(nodes, max_tokens=1500):
    """ Relevance-based Truncation: truncate the list of nodes to fit within the max token limit."""
    selected = []
    total_tokens = 0

    for node in nodes:
        node_text = node.text
        tokens = count_tokens(node_text)

        if total_tokens + tokens > max_tokens:
            break

        selected.append(node)
        total_tokens += tokens

    return selected

@traceable()
def summarize_context(filtered_nodes):
    """Context Optimization: summarize and compress  the context if it exceeds token limits."""
    total_tokens = sum(count_tokens(n.text) for n in filtered_nodes )

    if total_tokens > 1000:
        print("🧠 Applying summarization...")
        context = summarize_context(filtered_nodes)
    else:
        context = "\n\n".join([n.text for n in filtered_nodes])
    return context

In [ ]:

# ─────────────────────────────────────────────────────────
# HYBRID RAG + LONG-CONTEXT WINDOW ARCHITECTURE
# ─────────────────────────────────────────────────────────
# Strategy
#   • Short queries  → standard RAG  (retrieve top-k chunks, answer)
#   • Deep-analysis  → Long-context  (stuff the FULL document into a
#                                     128k-context model and answer)
#   • Hybrid         → RAG finds the right document(s); long-context
#                      window does the deep reading inside each doc.
# ─────────────────────────────────────────────────────────

# Long-context model (gpt-4o supports 128k tokens)
llm_long = OpenAI(model="gpt-4o", temperature=0, max_tokens=4096)

def _group_nodes_by_doc(nodes: list[NodeWithScore]) -> dict[str, list[NodeWithScore]]:
    """Group retrieved chunks back to their source document."""
    groups = defaultdict(list)
    for n in nodes:
        doc_id = n.node.ref_doc_id or n.node.node_id
        groups[doc_id].append(n)
    return groups

def hybrid_rag_long_context(
    query: str,
    top_k: int = 5,
    long_context_threshold: int = 3,   # if ≥N chunks from same doc → use long-context
    verbose: bool = True,
) -> str:
    """
    Hybrid architecture:
      1. RAG retrieves top-k chunks (broad knowledge).
      2. If multiple chunks come from the same source doc, pull the
         FULL document text and hand it to a long-context model
         for deep analysis.
      3. Otherwise fall back to standard chunk-level answer.
    """
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = vector_retriever.retrieve(query)

    groups = _group_nodes_by_doc(nodes)
    deep_doc_ids = [
        doc_id for doc_id, ns in groups.items()
        if len(ns) >= long_context_threshold
    ]

    if deep_doc_ids and verbose:
        print(f"[LongCtx]  Deep-reading {len(deep_doc_ids)} document(s)")

    answers = []

    # --- Deep path: full-document long-context analysis ---
    for doc_id in deep_doc_ids:
        # Reconstruct full document text from all stored nodes of this doc
        doc_nodes = [
            n for n in all_nodes
            if (getattr(n, "ref_doc_id", None) or n.node_id) == doc_id
        ]
        if not doc_nodes:
            doc_nodes = [n.node for n in groups[doc_id]]

        # Sort by start_char_idx if available
        doc_nodes_sorted = sorted(
            doc_nodes,
            key=lambda n: getattr(n, "start_char_idx", 0) or 0
        )
        full_text = "\n".join(
            getattr(n, "text", getattr(n, "get_content", lambda: "")())
            for n in doc_nodes_sorted
        )

        prompt = f"""You are performing deep document analysis.
            Read the FULL document below and answer the question thoroughly.

            Document:
            {full_text[:110_000]}   # stay within 128k context

            Question: {query}
            Answer:
            ans = llm_long.complete(prompt).text.strip()
            answers.append(f"[Deep analysis]\n{ans}")
            """

    # --- Broad path: standard RAG for remaining chunks ---
    remaining = [
        n for n in nodes
        if (n.node.ref_doc_id or n.node.node_id) not in deep_doc_ids
    ]
    if remaining:
        context = "\n\n".join(n.get_content() for n in remaining)
        prompt  = f"""Answer using ONLY the context below.
Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
        ans = llm.complete(prompt).text.strip()
        answers.append(f"[RAG answer]\n{ans}")

    return "\n\n---\n\n".join(answers) if answers else "No answer found."

In [ ]:
# -----------------------------
# LLM GENERATION
# -----------------------------

@traceable()
def generate_answer(query, context):
    """Generate an answer using the LLM based on the retrieved context."""
    answer_prompt = f"""Answer the question using ONLY the context below.


            Context:
            {context}

            Question: {query}
            Answer:"""


    # Step 5 — generate answer
    response = llm.complete(answer_prompt)
    return response.text.strip()

In [ ]:
# -----------------------------
# RUN AND STREAM ADAPTIVE RAG PIPELINE
# -----------------------------

from tabnanny import verbose
   
def adaptive_rag(query: str,
    metadata_filters: Optional[dict]        = None,
    use_long_context: bool = False,
    fusion_method:    str  = "rrf",   # "rrf" | "learned" | "dynamic" | "hyde"
    verbose:          bool  = True,
) -> str :
    """End-to-end adaptive RAG pipeline with query transformation, adaptive retrieval, re-ranking, and context optimization."""
    
    # Step 1 — classify and transform query
    cls = classify_query(query)
    if verbose:
        print(f"[Router]    {cls.query_type} — {cls.reasoning}")
    elif cls.query_type == "factual":
        retrieval_query = rewrite_query(query)
        if verbose: print(f"[Rewrite]   {retrieval_query}")
    elif cls.query_type in ("analytical", "conceptual"):
        sub_qs = decompose_query(query)
        retrieval_query = " ".join(sub_qs)   # join sub-questions for broader retrieval
        if verbose: print(f"[Decompose]  {sub_qs}")
    else:
        retrieval_query = query              # keep exact terms for BM25
        if verbose: print(f"[Keyword]    keeping original")
        
    retrieval_query, classification = classify_and_transform(query, verbose=verbose)
    
    # Step 2 — retrieve
    retriever = get_adaptive_retriever(classification.query_type)

    nodes = retriever.retrieve(retrieval_query)
    print(f"{Fore.CYAN}Stage 2 (🔍 Retrieval){Style.RESET_ALL}: {len(nodes)} nodes retrieved\n")
   

    # (replace with HyDE for factual)
   
    # nodes = retriever.retrieve(retrieval_query)
    
    # retrieve and rerank with different fusion methods based on query type
    
    # Step 3 — Context optimization    
            
    # Step 4 — Final : Context Assembly + LLM Generations

            

In [ ]:
if __name__ == "__main__":
    print("\n🚀 Running Query...\n")

    # 🎯 Queries aligned with Paul Graham essays corpus
    queries = [
        # Factual (retrieval-friendly)
        ("According to Paul Graham, what is a startup?", "rrf", False),

        # Conceptual / analytical (great for HyDE + decomposition)
        ("What does Paul Graham say about how startups should grow?", "learned", False),

        # Deep long-context analysis
        ("Summarize and analyze Paul Graham’s essay 'Do What You Love'", "dynamic", True),

        # Keyword-heavy (BM25-friendly)
        ("Paul Graham advice on startup funding and investors", "rrf", False),

        # Additional strong examples 👇
        ("What are the key characteristics of successful founders according to Paul Graham?", "learned", False),
        ("Why does Paul Graham emphasize building something people want?", "learned", False),
        ("Main lessons from Paul Graham essays about startups and growth", "dynamic", True),
        ("How does Paul Graham describe the early stage of a startup?", "learned", False),
]   
    
    for q, mode, long_ctx in queries:
        print(f"\n{'='*80}\nQuery: {q}\nMode: {mode}  Long-context: {long_ctx}\n{'='*80}")
        answer = adaptive_rag(q, fusion_method=mode, use_long_context=long_ctx, verbose=True)
        print(f"\nAnswer:\n{answer}\n")
    